# الدرس 03: بناء مسار RAG كامل مدمج به التقطيع الدلالي البرمجي المخصص
### Building an End-to-End RAG Pipeline with Custom Modular Semantic Chunker (Lecture 39)

مرحبًا بك في الدرس الثالث من مرحلة **تقنيات التقطيع الدلالي المتقدم (`Advanced Semantic Chunking`)**.

في هذا الدرس، ننتقل من مجرد كتابة دوال منفصلة إلى **البرمجة المعيارية كائنية التوجه (`Modular OOP`)**:
1. بناء كلاس مخصص قابل لإعادة الاستخدام: `ThresholdSemanticChunker`.
2. دعم التقطيع النصي `split_text` والتقطيع الوثائقي `split_documents` مع الحفاظ على البيانات الوصفية (`Metadata`).
3. فهرسة القطع الدلالية في قاعدة متجهات سريعة (`FAISS`).
4. بناء مسار استرجاع وتوليد متكامل بتعبيرات `LCEL` مع نموذج لغوي فائق السرعة عبر `Groq`.

---

In [1]:
# 1. استدعاء المكتبات وتهيئة البيئة
import os
import warnings
from dotenv import load_dotenv
from typing import List
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

warnings.filterwarnings("ignore")
load_dotenv()

print("✅ تم استدعاء كافة المكتبات وتهيئة مفاتيح البيئة بنجاح!")
print(f"مفتاح Groq متوفر: {bool(os.getenv('GROQ_API_KEY'))}")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_4520\2813713369.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ تم استدعاء كافة المكتبات وتهيئة مفاتيح البيئة بنجاح!
مفتاح Groq متوفر: True


---
## الخطوة 1: بناء كلاس التقطيع الدلالي المعياري (`ThresholdSemanticChunker Class`)

نقوم بإنشاء الكلاس المعياري كما تم شرحه بالمحاضرة:
- دالة البناء `__init__`: تستقبل اسم موديل التضمين وقيمة العتبة `threshold`.
- دالة `split_text`: تقسم النص إلى جمل، وتحسب المتجهات، وتطبق حلقة التجميع بالعتبة.
- دالة `split_documents`: تتعامل مع كائنات `Document` في LangChain وتحافظ على البيانات الوصفية `metadata` لكل وثيقة.

In [2]:
class ThresholdSemanticChunker:
    """
    محرك تقطيع دلالي مخصص ومعياري يعتمد على قياس تشابه الجمل المتجاورة
    وفصل القطع عند حدوث هبوط دلالي يقل عن العتبة المحددة.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2", threshold: float = 0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split_text(self, text: str) -> List[str]:
        # 1. تقسيم النص إلى جمل مستقلة ونقية
        sentences = [s.strip() for s in text.split("\n") if s.strip()]
        if not sentences:
            return []

        # 2. حساب المتجهات الكثيفة
        embeddings = self.model.encode(sentences)

        # 3. حلقة التجميع والمقارنة بالعتبة
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]

        if current_chunk:
            chunks.append(" ".join(current_chunk))

        return chunks

    def split_documents(self, docs: List[Document]) -> List[Document]:
        """تقطيع قائمة وثائق LangChain مع الحفاظ على البيانات الوصفية."""
        result = []
        for doc in docs:
            chunks = self.split_text(doc.page_content)
            for chunk in chunks:
                result.append(
                    Document(
                        page_content=chunk,
                        metadata=doc.metadata.copy()
                    )
                )
        return result

print("✅ تم تعريف الكلاس المعياري ThresholdSemanticChunker بنجاح!")

✅ تم تعريف الكلاس المعياري ThresholdSemanticChunker بنجاح!


---
## الخطوة 2: تجربة التقطيع الوثائقي (`Document Chunking`)

نجهز وثيقة تحتوي على نص متعدد الموضوعات ونمررها للـ `chunker`:

In [3]:
# تجهيز النص والوثيقة المرجعية من المحاضرة
with open("sample_text.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

source_doc = Document(
    page_content=raw_text,
    metadata={"source": "sample_text.txt", "author": "Course Lecture 39"}
)

# إنشاء كائن من الكلاس المعياري
chunker = ThresholdSemanticChunker(threshold=0.7)

# تنفيذ التقطيع الوثائقي
doc_chunks = chunker.split_documents([source_doc])

print(f"📊 إجمالي عدد القطع الوثائقية الناتجة: {len(doc_chunks)}\n")
for idx, d in enumerate(doc_chunks, 1):
    print(f"--- [وثيقة {idx}] (المصدر: {d.metadata.get('source')}) ---")
    print(f"{d.page_content}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📊 إجمالي عدد القطع الوثائقية الناتجة: 4

--- [وثيقة 1] (المصدر: sample_text.txt) ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.

--- [وثيقة 2] (المصدر: sample_text.txt) ---
You can create chains, agents, memory, and retrievers.

--- [وثيقة 3] (المصدر: sample_text.txt) ---
The Eiffel Tower is located in Paris.

--- [وثيقة 4] (المصدر: sample_text.txt) ---
France is a popular tourist destination.



---
## الخطوة 3: التخزين في مخزن المتجهات (`FAISS Vector Store`) وتجهيز المسترجع

نقوم بفهرسة القطع الدلالية داخل فهرس `FAISS` خفيف وسريع، ثم إنشاء المسترجع `retriever`:

In [4]:
# إعداد موديل التضمين للـ Vector Store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# بناء فهرس FAISS من القطع الدلالية
vector_store = FAISS.from_documents(doc_chunks, embeddings)

# إنشاء المسترجع الذكي
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# اختبار الاسترجاع لسؤال عن LangChain
test_query = "What is LangChain used for?"
retrieved_docs = retriever.invoke(test_query)

print(f"🔍 نتيجة استرجاع الوثائق للسؤال: '{test_query}':\n")
for i, r in enumerate(retrieved_docs, 1):
    print(f"[{i}] {r.page_content}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🔍 نتيجة استرجاع الوثائق للسؤال: 'What is LangChain used for?':

[1] LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
[2] You can create chains, agents, memory, and retrievers.


---
## الخطوة 4: بناء مسار الـ RAG بتعبيرات `LCEL` مع الموديل اللغوي

نقوم بدمج:
1. `RunnableParallel` لربط السؤال بالسياق المسترجع تلقائياً.
2. `ChatPromptTemplate` لصياغة القالب التوجيهي.
3. `ChatGroq` كنموذج لغوي سريع ومجاني.
4. `StrOutputParser` لاستخراج النص النهائي الصافي.

In [5]:
# 1. إعداد القالب التوجيهي
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context:
{context}

Question:
{question}

Answer:""")

# 2. إعداد الموديل اللغوي عبر Groq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.4)

# 3. تجميع سلسلة LCEL
rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever,
        question=lambda x: x["question"]
    )
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ تم بناء مسار RAG Chain بتعبيرات LCEL بنجاح!")

✅ تم بناء مسار RAG Chain بتعبيرات LCEL بنجاح!


---
## الخطوة 5: تنفيذ الاستعلامات والتحقق من دقة النتائج (`Query Execution & Verification`)

نقوم بتنفيذ استعلامين مختلفين للتأكد من عزل السياق الدلالي وعدم حدوث أي تشتت أو هلوسة:

In [6]:
# الاستعلام الأول: سؤال المحاضرة عن LangChain
q1 = "What is LangChain used for?"
res1 = rag_chain.invoke({"question": q1})

print(f"❓ السؤال 1: {q1}")
print(f"💡 إجابة مسار الـ RAG:\n{res1}\n")
print("=" * 60)

# الاستعلام الثاني: سؤال عن باريس وفرنسا
q2 = "Where is the Eiffel Tower located and what is France known for?"
res2 = rag_chain.invoke({"question": q2})

print(f"\n❓ السؤال 2: {q2}")
print(f"💡 إجابة مسار الـ RAG:\n{res2}")

❓ السؤال 1: What is LangChain used for?
💡 إجابة مسار الـ RAG:
LangChain is a framework for building applications that use large language models (LLMs). It offers modular abstractions that let developers combine LLMs with tools such as OpenAI and Pinecone, and provides components for creating chains, agents, memory, and retrievers.




❓ السؤال 2: Where is the Eiffel Tower located and what is France known for?
💡 إجابة مسار الـ RAG:
The Eiffel Tower is located in **Paris**, and **France is known for being a popular tourist destination**.


---
## الخطوة 6: الاختبارات البرمجية الصارمة (`Automated Assertions`)

In [7]:
# التحقق من دقة الاسترجاع وعزل السياق
assert len(doc_chunks) == 4, "يجب أن ينتج الكلاس 4 قطع دلالية عند العتبة 0.7"
assert "framework" in res1.lower(), "إجابة لانج تشين يجب أن تحتوي على وصف إطار العمل"
assert "paris" in res2.lower(), "إجابة برج إيفل يجب أن تذكر باريس"

print("✅ كافة الاختبارات البرمجية اجتازت بنجاح!")

✅ كافة الاختبارات البرمجية اجتازت بنجاح!
